In [ ]:
import os, re
root_data_dir = os.environ["ALGONAUTS_ROOT_DIR"]
acc_dir= f"{root_data_dir}/ann_brain_data/outputs"

import numpy as np
from tqdm.auto import tqdm
from matplotlib import pyplot as plt

from brainannlib.algonauts_funcs import compute_encoding_accuracy
from brainannlib.algonauts_funcs import plot_accuracy_on_brain
from brainannlib.stats_and_metrics import corr_score

excluded_samples_start, excluded_samples_end = 5, 5

# Remove pplots and just provide example code ...

# Result Overview

In [ ]:
"""
# Parcellated subject to subject (inidividual subject pairs in brackets):
friends-s01                     0.137 	 [0.151 0.128 0.134] 	 (3, 1000)
friends-s02                     0.147 	 [0.156 0.144 0.141] 	 (3, 1000)
friends-s03                     0.144 	 [0.162 0.134 0.136] 	 (3, 1000)
friends-s04                     0.148 	 [0.157 0.147 0.141] 	 (3, 1000)
friends-s05                     0.147 	 [0.165 0.141 0.134] 	 (3, 1000)
friends-s06                     0.141 	 [0.146 0.135 0.142] 	 (3, 1000)
movie10-bourne                  0.124 	 [0.14  0.117 0.115] 	 (3, 1000)
movie10-wolf                    0.146 	 [0.148 0.145 0.143] 	 (3, 1000)

movie10-figures[0-9]+_run-1     0.139 	 [0.146 0.142 0.13 ] 	 (3, 1000)
movie10-figures[0-9]+_run-2     0.135 	 [0.135 0.139 0.13 ] 	 (3, 1000)
movie10-life[0-9]+_run-1        0.099 	 [0.106 0.085 0.105] 	 (3, 1000)
movie10-life[0-9]+_run-2        0.111 	 [0.12  0.111 0.103] 	 (3, 1000)


# Parcellated test-retest (per subject:   sub01,sub03,sub05)
test-retest-figures:	        0.119    [0.132 0.123 0.103]
test-retest-life:       	    0.165    [0.157 0.21  0.129]

test-retest-life sub-01:	 0.157
test-retest-life sub-02:	 0.143
test-retest-life sub-03:	 0.21
test-retest-life sub-05:	 0.129
test-retest-figures sub-01:	 0.132
test-retest-figures sub-02:	 0.114
test-retest-figures sub-03:	 0.123
test-retest-figures sub-05:	 0.103

"""
# -> subject to subject correlation higher for parcellated
# -> within subject test-retest higher for full brain

In [ ]:
"""
# Sub-01 combined model trained on friends s1-5, parcellated test:
friends-s06      0.221    test
movie10-bourne   0.132    test
movie10-wolf     0.164    test
movie10-figures  0.199    test
movie10-life     0.13    test
"""

In [44]:
def prep_mri_pair(fmri1, fmri2, excluded_samples_start, excluded_samples_end, stim_sets, \
             v=False, n_targets=1000, custom_episodes=[]):
    
    # initialize empty array with 1000 parcels
    # this will return a matrix of concatenated timepoints across episodes/movies x 1000 parcels
    aligned_fmri1 = np.empty((0, n_targets), dtype=np.float32) 
    aligned_fmri2 = np.empty((0, n_targets), dtype=np.float32) 
    # the feature matrix will have the same # of timepoints as the aligned_fmri matrix

    isfirst = True; # just for verbose output/debugging

    for stimset in stim_sets:
        # stimset e.g. "s01", "s02", ... "bourne", "life", ...
        # but also acommodating their style: "friends-s01", "movie10-bourne" ...
        if stimset == "custom_episodes":
            episodes_in_set = custom_episodes;
        else:
            stimset = "-".join(stimset.split("-")[1:])
            #print(stimset)
            #episodes_in_set = [key for key in fmri1 if key.startswith(stimset)]
            episodes_in_set = [key for key in fmri1 if re.match(stimset, key)]
            episodes_in_set = [key for key in episodes_in_set if key in fmri2.keys()]
            difference  = list(set(fmri1.keys()) - set(fmri2.keys()))
            if v and (len(difference)>0): print("Missing episodes for 1 subj:", difference)

        if v>=1: print(stimset, len(episodes_in_set), episodes_in_set[:3])

        for episode in episodes_in_set:
            fmri_run1 = fmri1[episode][excluded_samples_start:-excluded_samples_end]
            fmri_run2 = fmri2[episode][excluded_samples_start:-excluded_samples_end]
            
            if fmri_run1.shape[0]!=fmri_run2.shape[0]:
                print("TRs are off for clip: ", episode,  fmri_run1.shape[0], "and", fmri_run2.shape[0])
            
            n_trs = min(fmri_run1.shape[0], fmri_run2.shape[0])
            fmri_run1=fmri_run1[:n_trs]; fmri_run2=fmri_run2[:n_trs];
            #print(n_trs, aligned_fmri.shape, fmri_run.shape )
            aligned_fmri1 = np.append(aligned_fmri1, fmri_run1, 0)
            aligned_fmri2 = np.append(aligned_fmri2, fmri_run2, 0)

            if isfirst and v>=2: print("\nsplit", episode, fmri_run1.shape)

            isfirst=False;

    return aligned_fmri1, aligned_fmri2

# Parcellated subject to subject

In [6]:
import os
from importlib import reload
import brainannlib.algonauts_funcs
reload(brainannlib.algonauts_funcs)

from brainannlib.algonauts_funcs import load_fmri

In [10]:
from brainannlib.algonauts_funcs import load_fmri

fmri_dict = {}
all_subs = ["sub-0"+str(s) for s in [1, 3,5]]
for s in [1, 3, 5]:
    sub = f"sub-0{s}"
    subj_fmri = load_fmri(root_data_dir, s, average_repeat_runs=False)
    fmri_dict[sub]=subj_fmri
    print(sub, len(fmri_dict[sub].keys()))

sub-01 353
sub-03 351
sub-05 350


In [ ]:
subj_perms = [[all_subs[a], all_subs[b]] for a,b in [[0,1],[0,2],[1,2]]]

from collections import defaultdict
scores_bw=defaultdict(list)
scores_al=defaultdict(list)

all_movie_sets = [ "friends-s01", "friends-s02", "friends-s03", "friends-s04", "friends-s05", \
                 "friends-s06", "movie10-bourne", "movie10-wolf"]# "movie10-figures", "movie10-life"]
all_movie_sets =all_movie_sets+ [r"movie10-figures[0-9]+_run-1", r"movie10-figures[0-9]+_run-2", r"movie10-life[0-9]+_run-1", r"movie10-life[0-9]+_run-2"]


for i, (sub_a, sub_b) in enumerate(subj_perms):

    for test in tqdm(all_movie_sets):
        fmri_a, fmri_b = prep_mri_pair(fmri_dict[sub_a],fmri_dict[sub_b], \
              excluded_samples_start, excluded_samples_end, [test], n_targets=1000)
        print(fmri_a.shape, fmri_b.shape)
        # correlate the score
        score_bw=corr_score(fmri_a, fmri_b)
        encoding_accuracy, mean_encoding_accuracy = compute_encoding_accuracy(fmri_a, fmri_b, -1, "None")

        # scores should be the same irrespective of the method
        if not np.allclose(score_bw, encoding_accuracy):
            print("Off: ", encoding_accuracy.mean(), score_bw.mean())
        print(sub_a, sub_b, "\t", test, "\t", score_bw.mean().round(3))
        
        scores_bw[test].append(score_bw)
        scores_al[test].append(encoding_accuracy)

In [37]:
fn=f"{acc_dir}/subj2subj_parcellated_baseline.npy"
np.save(fn, dict(scores=scores_bw, scores2=scores_al, subj_perms=subj_perms));

**show the results**

In [39]:
#fn=f"{acc_dir}/subj2subj_parcellated_baseline.npy"
#data = np.load(fn, allow_pickle=True).item()
data=dict(scores=scores_bw)

for movie_set, scores in data["scores"].items():
    scores=np.array(scores)
    print(movie_set, " "*(30-len(movie_set)), scores.mean(1).mean().round(3), "\t", scores.mean(1).round(3), "\t", scores.shape)

friends-s01                     0.137 	 [0.151 0.128 0.134] 	 (3, 1000)
friends-s02                     0.147 	 [0.156 0.144 0.141] 	 (3, 1000)
friends-s03                     0.144 	 [0.162 0.134 0.136] 	 (3, 1000)
friends-s04                     0.148 	 [0.157 0.147 0.141] 	 (3, 1000)
friends-s05                     0.147 	 [0.165 0.141 0.134] 	 (3, 1000)
friends-s06                     0.141 	 [0.146 0.135 0.142] 	 (3, 1000)
movie10-bourne                  0.124 	 [0.14  0.117 0.115] 	 (3, 1000)
movie10-wolf                    0.146 	 [0.148 0.145 0.143] 	 (3, 1000)
movie10-figures[0-9]+_run-1     0.139 	 [0.146 0.142 0.13 ] 	 (3, 1000)
movie10-figures[0-9]+_run-2     0.135 	 [0.135 0.139 0.13 ] 	 (3, 1000)
movie10-life[0-9]+_run-1        0.099 	 [0.106 0.085 0.105] 	 (3, 1000)
movie10-life[0-9]+_run-2        0.111 	 [0.12  0.111 0.103] 	 (3, 1000)


In [ ]:
#fn=f"{acc_dir}/subj2subj_parcellated_baseline.npy"
#data = np.load(fn, allow_pickle=True).item()
data=dict(scores=scores_bw)

for movie_set, scores in data["scores"].items():
    scores=np.array(scores)
    print(movie_set, " "*(20-len(movie_set)), scores.mean(1).mean().round(3), "\t", scores.mean(1).round(3), "\t", scores.shape)

friends-s01           0.137 	 [0.151 0.128 0.134] 	 (3, 1000)
friends-s02           0.147 	 [0.156 0.144 0.141] 	 (3, 1000)
friends-s03           0.144 	 [0.162 0.134 0.136] 	 (3, 1000)
friends-s04           0.148 	 [0.157 0.147 0.141] 	 (3, 1000)
friends-s05           0.147 	 [0.165 0.141 0.134] 	 (3, 1000)
friends-s06           0.141 	 [0.146 0.135 0.142] 	 (3, 1000)
movie10-bourne        0.124 	 [0.14  0.117 0.115] 	 (3, 1000)
movie10-wolf          0.146 	 [0.148 0.145 0.143] 	 (3, 1000)
movie10-figures       0.205 	 [0.205 0.212 0.198] 	 (3, 1000)
movie10-life          0.162 	 [0.168 0.15  0.167] 	 (3, 1000)


In [ ]:
# plotting 

for i, (movie_set, scores) in enumerate(data["scores"].items()):
    scores=np.array(scores)
    max=scores.mean(0).max().round(2);
    mean=scores.mean(1).mean().round(3);
    title="s2s:"+movie_set+" mean="+ str(mean)+", max="+str(max)
    # plotting using subject 1's parcellation, but scores reflect average across all 3 subject pairs
    plot_accuracy_on_brain(scores.mean(0),mean, 1, title, vmax=0.7);   
    plt.show()

# Parcellated test-tetest ceiling

In [107]:
from brainannlib.algonauts_funcs import load_full_fmri

fmri = load_full_fmri(root_data_dir, average_repeats=False)
ess,ese=5,5 # excluded_samples_start, excluded_samples_end

life1 = [k for k in fmri["sub-01"].keys() if ("life" in k) and ("_run-1" in k)]
life2 = [k for k in fmri["sub-01"].keys() if ("life" in k) and ("_run-2" in k)]
figs1 = [k for k in fmri["sub-01"].keys() if ("figures" in k) and ("_run-1" in k)]
figs2 = [k for k in fmri["sub-01"].keys() if ("figures" in k) and ("_run-2" in k)]
print(life1)

['life01_run-1', 'life02_run-1', 'life03_run-1', 'life04_run-1', 'life05_run-1']


In [109]:
for s in [1,2,3,5]:
    print(s, "life1:",[fmri[sub][epi].shape[0] for epi in life1])
    print(s,"life2:",[fmri[sub][epi].shape[0] for epi in life2])
    print(s,"figs1:",[fmri[sub][epi].shape[0] for epi in figs1])
    print(s,"figs2:",[fmri[sub][epi].shape[0] for epi in figs2])

1 life1: [406, 406, 406, 406, 384]
1 life2: [406, 406, 406, 406, 384]
1 figs1: [402, 409, 410, 408, 408, 408, 409, 409, 409, 409, 409, 373]
1 figs2: [402, 409, 410, 409, 408, 408, 408, 409, 409, 409, 409, 373]
2 life1: [406, 406, 406, 406, 384]
2 life2: [406, 406, 406, 406, 384]
2 figs1: [402, 409, 410, 408, 408, 408, 409, 409, 409, 409, 409, 373]
2 figs2: [402, 409, 410, 409, 408, 408, 408, 409, 409, 409, 409, 373]
3 life1: [406, 406, 406, 406, 384]
3 life2: [406, 406, 406, 406, 384]
3 figs1: [402, 409, 410, 408, 408, 408, 409, 409, 409, 409, 409, 373]
3 figs2: [402, 409, 410, 409, 408, 408, 408, 409, 409, 409, 409, 373]
5 life1: [406, 406, 406, 406, 384]
5 life2: [406, 406, 406, 406, 384]
5 figs1: [402, 409, 410, 408, 408, 408, 409, 409, 409, 409, 409, 373]
5 figs2: [402, 409, 410, 409, 408, 408, 408, 409, 409, 409, 409, 373]


In [ ]:
for s in [1,2,3,5]:
    sub=f"sub-0{s}"
    test = np.concatenate([fmri[sub][epi][ess:-ese] for epi in life1])
    retst = np.concatenate([fmri[sub][epi][ess:-ese] for epi in life2])
    ea, mea = compute_encoding_accuracy(test, retst, s, "test-retest")
    print("test-retest-life", sub+":\t", mea);

for s in [1,2,3,5]:
    sub=f"sub-0{s}"
    test = np.concatenate([fmri[sub][epi][ess:-ese] for epi in figs1])
    retst = np.concatenate([fmri[sub][epi][ess:-ese] for epi in figs2])
    ea, mea = compute_encoding_accuracy(test, retst, s, "test-retest")
    print("test-retest-figures", sub+":\t", mea);


test-retest-life sub-01:	 0.157
test-retest-life sub-02:	 0.143
test-retest-life sub-03:	 0.21
test-retest-life sub-05:	 0.129
test-retest-figures sub-01:	 0.132
test-retest-figures sub-02:	 0.114
test-retest-figures sub-03:	 0.123
test-retest-figures sub-05:	 0.103


In [ ]:
#plotting
for s in [1,2,3,5]:
    sub=f"sub-0{s}"
    test = np.concatenate([fmri[sub][epi][ess:-ese] for epi in life1])
    retst = np.concatenate([fmri[sub][epi][ess:-ese] for epi in life2])
    ea, mea = compute_encoding_accuracy(test, retst, s, "test-retest")
    plot_accuracy_on_brain(ea,mea, s, "test-retest-life", vmax=0.7);   

for s in [1,2,3,5]:
    sub=f"sub-0{s}"
    test = np.concatenate([fmri[sub][epi][ess:-ese] for epi in figs1])
    retst = np.concatenate([fmri[sub][epi][ess:-ese] for epi in figs2])
    ea, mea = compute_encoding_accuracy(test, retst, s, "test-retest")
    plot_accuracy_on_brain(ea,mea, s, "test-retest-figures", vmax=0.7);   
